## Supervised Learning

Reddit comment data will be used to build a Random Forest model that predicts the length of a message (Short, Medium, Long) given its subreddit ID, the weekday and the hour of the day.

Import libraries.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

Load Reddit data.

In [2]:
# Load Reddit comment data
df = pd.read_csv("data/anonymized/reddit/comments.csv")
print(df)

                     date    id  subreddit_id  body_length
0     2018-07-25 16:00:00     1             1           77
1     2018-07-27 13:00:00     2             1          287
2     2018-07-27 14:00:00     3             2          263
3     2018-07-27 16:00:00     4             1          307
4     2018-07-27 19:00:00     5             2           60
...                   ...   ...           ...          ...
3639  2026-03-22 16:00:00  3640           203           88
3640  2026-03-29 18:00:00  3641           179          245
3641  2026-03-30 06:00:00  3642           152           36
3642  2026-03-31 13:00:00  3643           179           97
3643  2026-03-31 17:00:00  3644           212           38

[3644 rows x 4 columns]


Normalize date columns and convert body_length into Short, Medium, Long categories.

In [3]:
# Normalize dates and create weekday/hour columns
df["date"] = pd.to_datetime(df["date"])
df["weekday"] = df["date"].dt.dayofweek
df["hour"] = df["date"].dt.hour

# Categorize the message lengths as short, medium, long (33.33%)
df['length_category'], bin_edges = pd.qcut(df['body_length'], q=3, labels=['Short', 'Medium', 'Long'], retbins=True)
print(f"length edges: {bin_edges}")

length edges: [1.000e+00 4.500e+01 1.190e+02 4.624e+03]


Convert subreddit IDs into one-hot encoding.

In [4]:
# Handle the Categorical Subreddit IDs (One-Hot Encoding)
# This creates a new True/False column for every subreddit
df = pd.get_dummies(df, columns=['subreddit_id'], drop_first=True)

Prepare features and expected results for supervised training. 

In [5]:
# Target (y) is what we want to predict: The message length
y = df['length_category']

# Features (X) are the inputs we use to make the prediction
# Drop columns that aren't features
X = df.drop(columns=['body_length', 'length_category', 'date', 'id'])

print(y)
print(X)

0       Medium
1         Long
2         Long
3         Long
4       Medium
         ...  
3639    Medium
3640      Long
3641     Short
3642    Medium
3643     Short
Name: length_category, Length: 3644, dtype: category
Categories (3, str): ['Short' < 'Medium' < 'Long']
      weekday  hour  subreddit_id_2  subreddit_id_3  subreddit_id_4  \
0           2    16           False           False           False   
1           4    13           False           False           False   
2           4    14            True           False           False   
3           4    16           False           False           False   
4           4    19            True           False           False   
...       ...   ...             ...             ...             ...   
3639        6    16           False           False           False   
3640        6    18           False           False           False   
3641        0     6           False           False           False   
3642        1    13  

Split data into training and test data.

In [6]:
# Split the data: 80% to train the model, 20% to test it on "unseen" data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Train a random forest model.

In [14]:
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

Evaluate the results.

In [13]:
# Have the model guess the categories for the unseen 20% of data
predictions = model.predict(X_test)

print("\n--- Model precision based on test data ---")
print(classification_report(y_test, predictions))



--- Model precision based on test data ---
              precision    recall  f1-score   support

        Long       0.44      0.49      0.47       231
      Medium       0.36      0.37      0.36       243
       Short       0.49      0.44      0.47       255

    accuracy                           0.43       729
   macro avg       0.43      0.43      0.43       729
weighted avg       0.43      0.43      0.43       729



In [11]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\n--- Features that are most important for determining the comment length ---")
print(importances.head(5))


--- Features that are most important for determining the comment length ---
hour                0.386682
weekday             0.177826
subreddit_id_170    0.017550
subreddit_id_179    0.016464
subreddit_id_14     0.013620
dtype: float64


**Conclusions:**
- Hour of the day is a more important predictor for the length of a comment than the weekday.
- If a comment was published on subreddit 170, this is a bigger predictor for the comment length compared to other subreddits. This seems accurate, since subreddit 170 corresponds to a subreddit where I almost always post very short comments, and it is a subreddit I have been very active in lately.
- The model achieved an accuracy of 43%. This is better than if the model was randomly picking lengths (33%), so it indicates that knowing the features of a comment does increase the odds of correctly predicting its length. Knowing more features of comments (reply to post vs. reply to comment, number of paragraphs, depth of a comment in a comment chain) could have helped with increasing this accuracy, but these features aren't available in the anonymized data.